In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [5]:
len(words)

32033

In [11]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)


{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [27]:
block_size = 3
def build_dataset(words):
    X, Y = [], []
    
    for w in words:
        context = [0] * block_size 
        
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X,Y

import random
random.seed(42)
random.shuffle(words)
n1 =  int(0.8 *len(words))
n2 = int(0.90 *len( words))
Xtrain , Ytrain = build_dataset(words[:n1])
Xdev , Ydev = build_dataset(words[n1:n1])
Xtest , Ytest = build_dataset(words[n2:])

In [28]:
n_embd = 10 #the dimensitionality of character embedding vectors
n_hidden = 200 #the number of neuron in hidden layer of mlp

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size , n_embd),generator = g)
W1 = torch.randn(( n_embd * block_size , n_hidden), generator= g)
b1 = torch.randn(n_hidden, generator = g)
W2 = torch.randn((n_hidden,vocab_size), generator = g)
b2 = torch.randn(vocab_size , generator = g)


parameters = [C,W1,b1,W2,b2]
print(sum(p.nelement() for p in parameters))

for p in parameters:
    p.requires_grad = True



11897


In [ ]:
max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):
    
    #minibatch construct
    ix = torch.randint(0 , Xtrain.shape[0] , (batch_size,),generator = g)
    Xb,Yb= Xtrain[ix] , Ytrain[ix]
    
    #fordward pass
    emb = C[Xb] #embed the characters into vectors
    embcat = emb.view(emb.shape[0],-1) #concatinate the vectors
    hpreact = embcat @ W1 + b1 #hidden layer for preactivation
    h = torch.tanh(hpreact) #hidden layer
    logits = h @ W2 + b2 #outputlayer
    loss = F.cross_entropy(logits,Yb) #loss function

    #backwardpass
    for p in parameters:
        p.grad = None
    loss.backward()


    #update gradients
    lr = 0.1 if i < 100000 else 0.01 #step learning rate decay
    for p in parameters:
        p.data +=  lr * -p.grad

    #trackstats
    if i%10000 == 0: #print enery once in a while
        print(f'{i:7d}/{max_steps:7d} : {loss.item():.4f}')
    lossi.append(loss.log10().item())
    
    



      0/ 200000 : 23.7292
  10000/ 200000 : 2.0135
  20000/ 200000 : 2.4666
  30000/ 200000 : 2.5550
  40000/ 200000 : 2.3120
  50000/ 200000 : 1.8795
  60000/ 200000 : 2.4376
  70000/ 200000 : 2.4216
  80000/ 200000 : 2.7848
  90000/ 200000 : 2.3644
 100000/ 200000 : 2.1047
 110000/ 200000 : 2.0352
 120000/ 200000 : 2.3267
